# Labwork 1 - PCA: Raisin dataset
**Full name:** ...  |  **Student ID:** ...

This notebook covers Part 1 (Study the dataset) and Part 2 (PCA) for the **Raisin** dataset (UCI id 850).
Every code cell is followed by an *Explanation* cell to make it easy to follow.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")

**Explanation:** `pandas`/`numpy` handle the data, `matplotlib`/`seaborn` draw the plots, `StandardScaler` standardizes the features, and `PCA` is the main algorithm of this labwork.

## 1. Dataset description

In [ ]:
raisin = fetch_ucirepo(id=850)          # download the Raisin data from UCI
X = raisin.data.features                # feature columns
y = raisin.data.targets                 # label column (Class)
df = pd.concat([X, y], axis=1)          # merge into a single table

df.to_csv("../data/raw/raisin_raw.csv", index=False)   # save the raw copy
print(df.shape)
df.head()

**Explanation:** `fetch_ucirepo(id=850)` downloads the data automatically. We merge the features and the label into one table `df` and save the raw copy in `data/raw/`.

**Description (to complete):** Raisin contains 900 samples of two raisin varieties (Kecimen and Besni); each sample has 7 shape features measured from images. *(TODO: write 2-3 sentences in your own words and cite the UCI source.)*

## 2. Feature types

In [ ]:
print(df.dtypes)

**Explanation:** `dtypes` shows the data type of each column.

**TODO - fill in the classification table:**

| Feature | Discrete / Continuous | Quantitative / Qualitative | Numerical / Categorical |
|---|---|---|---|
| Area | ? | ? | ? |
| MajorAxisLength | ? | ? | ? |
| MinorAxisLength | ? | ? | ? |
| Eccentricity | ? | ? | ? |
| ConvexArea | ? | ? | ? |
| Extent | ? | ? | ? |
| Perimeter | ? | ? | ? |
| Class | ? | ? | ? |

## 3. Data quality issues

In [ ]:
print("Missing values:")
print(df.isna().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
df.describe()

**Explanation:** `isna().sum()` counts missing values per column; `duplicated().sum()` counts duplicate rows; `describe()` shows min/max/mean so that abnormal values (for example negative values in an area column) can be spotted.

In [ ]:
# Detect outliers with the IQR rule
num_cols = df.select_dtypes(include="number").columns

outlier_count = {}
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_count[col] = ((df[col] < low) | (df[col] > high)).sum()

pd.Series(outlier_count, name="n_outliers")

**Explanation:** a value is an outlier if it lies outside [Q1 - 1.5·IQR, Q3 + 1.5·IQR]. The `for` loop counts the outliers of each column.

**TODO - discussion:** list the issues found (if there are no missing values or duplicates, state clearly "checked, none found") and explain how they affect the analysis (outliers distort the mean and variance, affect PCA, ...).

In [ ]:
# Boxplots to visualize outliers
df[num_cols].plot(kind="box", subplots=True, layout=(2, 4), figsize=(14, 7))
plt.tight_layout()
plt.savefig("../figures/raisin_boxplot_before.png", dpi=150)
plt.show()

**Explanation:** each box is one feature and the dots outside the whiskers are outliers. The figure is saved in `figures/` so it can be inserted into the report.

## 4. Data cleaning

In [ ]:
df_clean = df.copy()

# 4.1 Remove duplicate rows
df_clean = df_clean.drop_duplicates()

# 4.2 Remove rows with missing values (if any)
df_clean = df_clean.dropna()

# 4.3 Handle outliers: only CLIP them to the IQR bounds instead of deleting samples
for col in num_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    df_clean[col] = df_clean[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

print("Before:", df.shape, "| After:", df_clean.shape)
df_clean.to_csv("../data/processed/raisin_clean.csv", index=False)

**Explanation:** each step handles one problem. `clip` keeps the number of samples but pulls outliers back to the bounds, which is safe when the dataset is small (900 samples).

**TODO:** compare before/after (number of rows, mean and std of a few features) and explain why clipping was chosen instead of deleting.

In [ ]:
# Compare statistics before and after cleaning
comparison = pd.DataFrame({
    "mean_before": df[num_cols].mean(),
    "mean_after": df_clean[num_cols].mean(),
    "std_before": df[num_cols].std(),
    "std_after": df_clean[num_cols].std(),
})
comparison

## 5. Data integration
**TODO:** Raisin is a single table with no accompanying source. Write 3-4 sentences explaining why integrating additional data is unnecessary (or, if you found an extra source, describe how you merged it and the challenges you met).

## 6. Data transformation

In [ ]:
features = df_clean.drop(columns=["Class"])

# Standardization: every feature gets mean = 0 and std = 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# Label encoding: text -> number
labels = df_clean["Class"].astype("category")
y_encoded = labels.cat.codes
print(dict(enumerate(labels.cat.categories)))

pd.DataFrame(X_scaled, columns=features.columns).describe().round(2)

**Explanation:** features such as Area (tens of thousands) and Eccentricity (0-1) have very different scales, so they must be *standardized* before PCA. The label `Class` is text, so it is encoded as 0/1.

**TODO:** explain why standardization was chosen instead of min-max scaling.

## 7. Label and machine learning task
**TODO:** the label is `Class` (Kecimen / Besni): the raisin variety. This is a supervised **binary classification** task. Add its practical meaning.

In [ ]:
df_clean["Class"].value_counts()

**Explanation:** count the samples of each class to check whether the data is balanced.

## 8. Mean, variance, covariance, correlation

In [ ]:
stats = pd.DataFrame({
    "mean": features.mean(),
    "variance": features.var(),
})
print(stats)

print("\nCovariance matrix:")
cov = features.cov()
print(cov.round(2))

corr = features.corr()

**Explanation:** `var()` is the variance, `cov()` is the covariance matrix, and `corr()` is the Pearson correlation matrix (scaled to -1...1, so it is easy to compare).

**About categorical features:** Raisin has only one categorical column, `Class`. TODO: describe the suitable analysis (frequency table, or comparing the mean of a feature across the classes with boxplots / a t-test).

## 9. Strongest correlated pair of features

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Raisin - correlation matrix")
plt.tight_layout()
plt.savefig("../figures/raisin_corr_heatmap.png", dpi=150)
plt.show()

# Find the strongest pair (ignore the diagonal)
pairs = corr.abs().where(~np.eye(len(corr), dtype=bool)).stack()
best_pair = pairs.idxmax()
print("Strongest pair:", best_pair, "-> r =", round(corr.loc[best_pair], 3))

**Explanation:** in the heatmap red means positive correlation and blue means negative. `where(~np.eye(...))` removes the diagonal (every feature has correlation 1 with itself) and `idxmax()` picks the largest pair.

**TODO:** comment on the heatmap (which groups of features are highly correlated and why, e.g. Area and ConvexArea both measure size, so they are almost linear).

## 10. Summary of Part 1
**TODO:** summarize the main findings, and explain how cleaning / integration / transformation affect the quality and reliability of the analysis.

# Part 2: PCA

## 11. Applying PCA

In [ ]:
pca = PCA()                     # keep all components
X_pca = pca.fit_transform(X_scaled)

print("Number of components:", X_pca.shape[1])

**Explanation:** the data is already standardized (a mandatory preprocessing step). PCA finds the directions (principal components) along which the data varies the most; in essence they are the eigenvectors of the covariance matrix, sorted by decreasing eigenvalue.

**TODO:** write 3-4 sentences explaining how the PCs are computed (covariance -> eigenvalues/eigenvectors) and how they are selected.

## 12. Explained variance

In [ ]:
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)

table = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(evr))],
    "explained_variance_ratio": evr.round(4),
    "cumulative": cum.round(4),
})
print(table)

plt.figure(figsize=(7, 4))
plt.bar(range(1, len(evr) + 1), evr)
plt.plot(range(1, len(evr) + 1), cum, marker="o", color="red")
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title("Raisin - explained variance")
plt.tight_layout()
plt.savefig("../figures/raisin_explained_variance.png", dpi=150)
plt.show()

**Explanation:** `explained_variance_ratio_` is the share of information (variance) kept by each PC. The red line is the cumulative variance.

**TODO:** compare the PC with the highest variance to the one with the lowest: the first PCs keep the main structure, the last ones are usually mostly noise.

## 13. Varying the number of components

In [ ]:
for k in range(1, X_scaled.shape[1] + 1):
    p = PCA(n_components=k)
    Xk = p.fit_transform(X_scaled)
    X_back = p.inverse_transform(Xk)                 # reconstruct the data
    error = np.mean((X_scaled - X_back) ** 2)        # reconstruction error
    print(f"k={k}: keeps {p.explained_variance_ratio_.sum():.1%} of the variance, reconstruction error = {error:.4f}")

**Explanation:** for each number of components k we check how much information is kept and the error made when rebuilding the original data from k components.

**TODO:** comment on the smallest k that reaches, for example, 95% of the variance and on the trade-off between dimensionality reduction and information loss.

## 14. 2D visualization (PC1 vs PC2)

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df_clean["Class"].values, alpha=0.7)
plt.xlabel(f"PC1 ({evr[0]:.1%})")
plt.ylabel(f"PC2 ({evr[1]:.1%})")
plt.title("Raisin - PCA 2D")
plt.tight_layout()
plt.savefig("../figures/raisin_pca_2d.png", dpi=150)
plt.show()

**Explanation:** each point is a sample, colored by `Class`. If the two colors are separated, PC1/PC2 capture the information that distinguishes the classes.

**TODO:** comment on the clusters and the overlapping regions.

In [ ]:
# Loadings: how much each feature contributes to PC1 and PC2 (helps interpretation)
loadings = pd.DataFrame(pca.components_[:2].T, index=features.columns, columns=["PC1", "PC2"])
loadings.round(3)

**Explanation:** the loadings show which features each PC is a linear combination of; use them to discuss interpretability.

## 15. Original data vs PCA representation
**TODO:** discuss
- **Advantages:** dimensionality reduction, decorrelated features, easy visualization.
- **Limitations:** some information is lost, PCs are harder to interpret than the original features, only linear relationships are captured, sensitivity to outliers and scale.